# COSETTE & MARIUS — Amazon 2014 replication

Table 5 for **Beauty** and **Sports & Outdoors** (Lepage et al.): pipeline checklist, test R@K / NDCG@K (mean ± std over seeds 42–46), and validation curves from `reports/`.

**Training:** Slurm jobs in `jobs/` (`jobs/RUN_ORDER.md`). Optional notebook commands in Section 0.

**Dependencies:** `jupyter`, `pandas`, `matplotlib`. Reads local files only (no WandB at runtime).

### Refreshing `reports/` after a new run

```bash
conda activate recsys
source jobs/wandb.env                    # only for --source wandb
export OUTPUT_ROOT=$SCRATCH/cosette_marius/outputs
python scripts/export_metrics_for_report.py --copy-results --source wandb
```

Without WandB: use `--source checkpoints` (fewer validation points).

**Paths:** uses `reports/results/` when present; otherwise `$OUTPUT_ROOT/results/`.


## 0. Replicating the experiments

See `jobs/RUN_ORDER.md` and `REPRODUCIBILITY.md`.

### Snellius (Slurm)

From the repo root, submit jobs **in order** for each dataset. Replace `/home/scur1266` with your home path (and check `#SBATCH --output` / `#SBATCH --error` in each `.sbatch` file).

**Setup**
```bash
cd /home/<USER>/rec_sys_cosette_marius
cp jobs/wandb.env.example jobs/wandb.env   # optional
conda activate recsys
export SCRATCH=/home/<USER>/scratch
```

**Beauty** (jobs 01→07; 1–3 SASRec++, 4–7 COSETTE → MARIUS)

| Step | Command | |
|------|---------|---|
| 1 | `sbatch jobs/01_download_beauty.sbatch` | Download |
| 2 | `sbatch jobs/02_parquet_beauty.sbatch` | Parquet |
| 3 | `sbatch jobs/03_sasrec_beauty_5seed_full.sbatch` | SASRec++ (~6–8 h) |
| 4 | `sbatch jobs/04_embeddings_beauty.sbatch` | Embeddings |
| 5 | `sbatch jobs/05_cosette_beauty.sbatch` | COSETTE |
| 6 | `sbatch jobs/06_remove_collisions_beauty.sbatch` | Collisions → `*-col` |
| 7 | `sbatch jobs/07_marius_beauty_5seed_full.sbatch` | MARIUS (~6–10 h) |

**Sports & Outdoors:** same with `*_sports.sbatch`.

Jobs 03 and 07 skip seeds already marked `"status": "ok"` in the scores jsonl.

### Export + notebook

```bash
source jobs/wandb.env
conda activate recsys
python scripts/export_metrics_for_report.py --copy-results --source wandb
jupyter notebook notebooks/replication_report.ipynb
```

### Optional: run from the notebook

The next cell lists the same commands as the job files (markdown, not executed). Copy into a code cell, set `RUN_PIPELINE = True` and `enabled=True` on the steps you need, and run on a GPU node.


### Optional pipeline commands

Not executed when running the notebook. Copy the block below into a code cell to run steps interactively on a GPU node.

```python
# Replication pipeline (disabled by default)
#
# To re-run experiments from this notebook:
#   1. Set paths below (or export DATA_ROOT / OUTPUT_ROOT / PYTHON_BIN).
#   2. Set RUN_PIPELINE = True.
#   3. Set enabled=True on the steps you want (one at a time, or the full list).
#   4. Run this cell on a GPU compute node (not a login node).
#
# On Snellius, prefer: sbatch jobs/01_… through jobs/07_… (see markdown above).

from __future__ import annotations

import os
import subprocess
from pathlib import Path

USER = os.environ.get("USER", "<USER>")
REPO_ROOT = Path("/home") / USER / "rec_sys_cosette_marius"
SCRATCH = Path(os.environ.get("SCRATCH", f"/home/{USER}/scratch"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", SCRATCH / "cosette_marius/data"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", SCRATCH / "cosette_marius/outputs"))
PYTHON = os.environ.get("PYTHON_BIN", "python")

# --- toggle replication here ---
RUN_PIPELINE = False

# Replace COSETTE_XXXX with the quant id from step 05; use the -col id in step 07.
COSETTE_BEAUTY_ID = "COSETTE_128d_256x4_XXXX"
COSETTE_SPORTS_ID = "COSETTE_128d_256x4_XXXX"

# Each step: set enabled=True to run when RUN_PIPELINE is True.
PIPELINE_STEPS: list[dict] = [
    # Beauty
    {
        "name": "beauty 01 download",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/download_data.py --year 2014 --base_dir {DATA_ROOT} --categories Beauty",
    },
    {
        "name": "beauty 02 parquet",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && PYTHONPATH=. {PYTHON} data_scripts/0_raw_to_parquet.py --config-name 0_raw_to_parquet_2014 categories='[Beauty]' paths.skip_download=true paths.root={DATA_ROOT}",
    },
    {
        "name": "beauty 03 SASRec++ 5-seed",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/sasrec_5seed.py --mode full --category Beauty --category-slug beauty",
    },
    {
        "name": "beauty 04 embeddings",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} data_scripts/1_make_embeddings.py category=Beauty num_gpus=1 batch_size=64 paths.root={DATA_ROOT} model_folder=sentence-transformers/sentence-t5-xl",
    },
    {
        "name": "beauty 05 COSETTE",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} data_scripts/2_train_cosette.py data.category=Beauty num_gpus=1 optim.batch_size=256 optim.epochs=1000 optim.eval_step=100 optim.dropout_prob=0.1 paths.root={DATA_ROOT} ckpt_dir={OUTPUT_ROOT}/cosette",
    },
    {
        "name": "beauty 06 collision removal",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && PYTHONPATH=. {PYTHON} data_scripts/3_remove_colisions.py data.category=Beauty data.emb_method=sentence-t5-xl data.quant_method={COSETTE_BEAUTY_ID} paths.root={DATA_ROOT}",
    },
    {
        "name": "beauty 07 MARIUS 5-seed",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/marius_5seed.py --mode full --category Beauty --category-slug beauty --quant-id {COSETTE_BEAUTY_ID}-col",
    },
    # Sports (same order)
    {
        "name": "sports 01 download",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/download_data.py --year 2014 --base_dir {DATA_ROOT} --categories Sports_and_Outdoors",
    },
    {
        "name": "sports 02 parquet",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && PYTHONPATH=. {PYTHON} data_scripts/0_raw_to_parquet.py --config-name 0_raw_to_parquet_2014 categories='[Sports_and_Outdoors]' paths.skip_download=true paths.root={DATA_ROOT}",
    },
    {
        "name": "sports 03 SASRec++ 5-seed",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/sasrec_5seed.py --mode full --category Sports_and_Outdoors --category-slug sports",
    },
    {
        "name": "sports 04 embeddings",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} data_scripts/1_make_embeddings.py category=Sports_and_Outdoors num_gpus=1 batch_size=64 paths.root={DATA_ROOT} model_folder=sentence-transformers/sentence-t5-xl",
    },
    {
        "name": "sports 05 COSETTE",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} data_scripts/2_train_cosette.py data.category=Sports_and_Outdoors num_gpus=1 optim.batch_size=256 optim.epochs=1000 optim.eval_step=100 optim.dropout_prob=0.1 paths.root={DATA_ROOT} ckpt_dir={OUTPUT_ROOT}/cosette",
    },
    {
        "name": "sports 06 collision removal",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && PYTHONPATH=. {PYTHON} data_scripts/3_remove_colisions.py data.category=Sports_and_Outdoors data.emb_method=sentence-t5-xl data.quant_method={COSETTE_SPORTS_ID} paths.root={DATA_ROOT}",
    },
    {
        "name": "sports 07 MARIUS 5-seed",
        "enabled": False,
        "cmd": f"cd {REPO_ROOT} && {PYTHON} scripts/marius_5seed.py --mode full --category Sports_and_Outdoors --category-slug sports --quant-id {COSETTE_SPORTS_ID}-col",
    },
]

print("Replication paths:")
print(f"  REPO_ROOT   = {REPO_ROOT}")
print(f"  DATA_ROOT   = {DATA_ROOT}")
print(f"  OUTPUT_ROOT = {OUTPUT_ROOT}")
print(f"  RUN_PIPELINE = {RUN_PIPELINE}")
print("\nSteps (enabled flag):")
for step in PIPELINE_STEPS:
    flag = "RUN" if step["enabled"] else "skip"
    print(f"  [{flag}] {step['name']}")

if not RUN_PIPELINE:
    print("\nSet RUN_PIPELINE=True and enabled=True on steps to execute.")
    print("On Snellius, sbatch jobs/01–07 is recommended instead.")
else:
    for step in PIPELINE_STEPS:
        if not step["enabled"]:
            continue
        print(f"\n>>> {step['name']}\n{step['cmd']}\n")
        subprocess.run(step["cmd"], shell=True, check=True, executable="/bin/bash")
    print("\nDone. Export results: python scripts/export_metrics_for_report.py --copy-results [--source wandb]")
```


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.reporting.constants import (
    CATEGORIES,
    EXPECTED_SEEDS,
    METRIC_COLUMNS,
    PAPER_MARIUS_COSETTE,
    PAPER_SASREC_PP,
)
from scripts.reporting.loaders import (
    load_scores_jsonl,
    pipeline_status,
    summarize_method,
)

SCRATCH_DEFAULT = Path(os.environ.get("SCRATCH", "/home/scur1266/scratch"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", SCRATCH_DEFAULT / "cosette_marius/outputs"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", SCRATCH_DEFAULT / "cosette_marius/data"))

REPORTS_DIR = REPO_ROOT / "reports"
METRICS_DIR = REPORTS_DIR / "metrics"
COMMITTED_RESULTS = REPORTS_DIR / "results"
LIVE_RESULTS = OUTPUT_ROOT / "results"

if LIVE_RESULTS.exists():
    RESULTS_DIR = LIVE_RESULTS
    print(f"Results: {RESULTS_DIR}")
elif COMMITTED_RESULTS.exists() and any(COMMITTED_RESULTS.glob("*.jsonl")):
    RESULTS_DIR = COMMITTED_RESULTS
    print(f"Results: {RESULTS_DIR}")
else:
    RESULTS_DIR = COMMITTED_RESULTS
    print(
        "No score files found. Export results with:\n"
        "  python scripts/export_metrics_for_report.py --copy-results"
    )

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


def load_metrics_csvs() -> pd.DataFrame:
    frames = []
    for path in sorted(METRICS_DIR.glob("*.csv")):
        if path.name == ".gitkeep":
            continue
        df = pd.read_csv(path)
        df["source_file"] = path.name
        frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

## 1. Pipeline status

In [ ]:
status_rows = pipeline_status(
    data_root=DATA_ROOT,
    output_root=OUTPUT_ROOT if OUTPUT_ROOT.exists() else None,
    results_dir=RESULTS_DIR,
)
status_df = pd.DataFrame(status_rows)
status_df["✓"] = status_df["status"].map({"done": "✓", "pending": "·"})
display(status_df[["✓", "label", "detail"]])

## 2. Table 5: test metrics (paper vs replication, mean ± std)

In [ ]:
def fmt_mean_std(mean: float, std: float) -> str:
    return f"{mean:.2f}% ±{std:.2f}"


def build_table5_rows(category: str) -> pd.DataFrame:
    slug = CATEGORIES[category]["slug"]
    rows = []

    configs = [
        ("SASRec++ (paper)", PAPER_SASREC_PP[category], None),
        (
            "SASRec++ (replication)",
            PAPER_SASREC_PP[category],
            load_scores_jsonl(RESULTS_DIR / f"sasrec_{slug}_5seed_full_scores.jsonl"),
        ),
        ("MARIUS + COSETTE (paper)", PAPER_MARIUS_COSETTE[category], None),
        (
            "MARIUS + COSETTE (replication)",
            PAPER_MARIUS_COSETTE[category],
            load_scores_jsonl(RESULTS_DIR / f"marius_{slug}_5seed_full_scores.jsonl"),
        ),
    ]

    for method, paper, records in configs:
        row = {"Method": method, "Seeds": "n/a" if records is None else len(records)}
        if records is None:
            for metric in METRIC_COLUMNS:
                mean, std = paper[metric]
                row[metric] = fmt_mean_std(mean, std)
        else:
            summary = summarize_method(records)
            if summary is None:
                for metric in METRIC_COLUMNS:
                    row[metric] = "pending"
            else:
                for metric in METRIC_COLUMNS:
                    row[metric] = fmt_mean_std(
                        summary[f"{metric}_mean"], summary[f"{metric}_std"]
                    )
        rows.append(row)
    return pd.DataFrame(rows)


for category in CATEGORIES:
    print(f"\n=== Amazon 2014 {category} (test, 5 seeds: {EXPECTED_SEEDS}) ===")
    display(build_table5_rows(category))

In [ ]:
def per_seed_table(category: str, method: str, filename: str) -> pd.DataFrame | None:
    records = load_scores_jsonl(RESULTS_DIR / filename)
    if not records:
        print(f"{method} {category}: no scores yet ({filename})")
        return None
    rows = []
    for record in sorted(records, key=lambda r: int(r["seed"])):
        row = {"seed": record["seed"], "run": record.get("run_directory", "")}
        for metric in METRIC_COLUMNS:
            row[metric] = f"{float(record[metric]):.2f}%"
        rows.append(row)
    return pd.DataFrame(rows)


for category, meta in CATEGORIES.items():
    slug = meta["slug"]
    print(f"\nPer-seed results: {category}")
    display(per_seed_table(category, "SASRec++", f"sasrec_{slug}_5seed_full_scores.jsonl"))
    display(per_seed_table(category, "MARIUS (COSETTE)", f"marius_{slug}_5seed_full_scores.jsonl"))

## 3. Paper vs replication (R@10 / NDCG@10)

In [ ]:
def plot_paper_vs_replication(category: str) -> None:
    slug = CATEGORIES[category]["slug"]
    plot_metrics = ["R@10", "NDCG@10"]
    methods = [
        ("SASRec++", PAPER_SASREC_PP[category], f"sasrec_{slug}_5seed_full_scores.jsonl"),
        ("MARIUS (COSETTE)", PAPER_MARIUS_COSETTE[category], f"marius_{slug}_5seed_full_scores.jsonl"),
    ]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=False)
    for ax, metric in zip(axes, plot_metrics):
        labels, paper_vals, repl_vals, repl_errs = [], [], [], []
        for label, paper, fname in methods:
            records = load_scores_jsonl(RESULTS_DIR / fname)
            summary = summarize_method(records)
            labels.append(label)
            paper_vals.append(paper[metric][0])
            if summary is None:
                repl_vals.append(0)
                repl_errs.append(0)
            else:
                repl_vals.append(summary[f"{metric}_mean"])
                repl_errs.append(summary[f"{metric}_std"])

        x = range(len(labels))
        width = 0.35
        ax.bar([i - width / 2 for i in x], paper_vals, width, label="Paper", color="#4C72B0")
        ax.bar(
            [i + width / 2 for i in x],
            repl_vals,
            width,
            yerr=repl_errs,
            capsize=4,
            label="Replication",
            color="#DD8452",
        )
        ax.set_xticks(list(x), labels)
        ax.set_ylabel(f"{metric} (%)")
        ax.set_title(metric)
        ax.legend()

    fig.suptitle(f"Amazon 2014 {category}, test set")
    fig.tight_layout()
    plt.show()


for category in CATEGORIES:
    plot_paper_vs_replication(category)

## 3b. Validation curves and gaps vs paper

Mean validation curves for SASRec++ and MARIUS (COSETTE), averaged over five seeds. Test-set gap vs Table 5 and per-seed spread.

In [ ]:
import numpy as np

metrics_df = load_metrics_csvs()


def aggregate_val_curve(
    df: pd.DataFrame,
    *,
    method: str,
    category: str,
    metric: str,
) -> pd.DataFrame | None:
    subset = df[
        (df["method"] == method)
        & (df["category"] == category)
        & (df["metric"] == metric)
    ]
    if subset.empty:
        return None
    wide = subset.pivot_table(index="step", columns="seed", values="value", aggfunc="mean")
    wide = wide.sort_index()
    return pd.DataFrame(
        {
            "step": wide.index.astype(int),
            "mean": wide.mean(axis=1).values,
            "std": wide.std(axis=1).values,
        }
    )


def plot_averaged_sasrec_vs_marius(category: str) -> None:
    if metrics_df.empty:
        print("No metrics CSVs; skip section 3b curves.")
        return

    method_styles = [
        ("sasrec", "SASRec++", "#4C72B0"),
        ("marius", "MARIUS (COSETTE)", "#DD8452"),
    ]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["HR@10", "NDCG@10"]):
        for method_key, label, color in method_styles:
            agg = aggregate_val_curve(metrics_df, method=method_key, category=category, metric=metric)
            if agg is None:
                continue
            y = 100.0 * agg["mean"]
            ystd = 100.0 * agg["std"].fillna(0.0)
            ax.plot(agg["step"], y, label=label, color=color, linewidth=2)
            ax.fill_between(agg["step"], y - ystd, y + ystd, color=color, alpha=0.2)
        ax.set_xlabel("Training step")
        ax.set_ylabel(f"validation {metric} (%)")
        ax.set_title(metric)
        ax.legend(fontsize=8)
    fig.suptitle(f"{category}: SASRec++ vs MARIUS (mean ± std over seeds)")
    fig.tight_layout()
    plt.show()


def plot_replication_gap(category: str) -> None:
    slug = CATEGORIES[category]["slug"]
    configs = [
        ("SASRec++", PAPER_SASREC_PP[category], f"sasrec_{slug}_5seed_full_scores.jsonl"),
        ("MARIUS (COSETTE)", PAPER_MARIUS_COSETTE[category], f"marius_{slug}_5seed_full_scores.jsonl"),
    ]
    rows = []
    for name, paper, fname in configs:
        summary = summarize_method(load_scores_jsonl(RESULTS_DIR / fname))
        if summary is None:
            continue
        for metric in ["R@5", "R@10", "NDCG@5", "NDCG@10"]:
            rep = summary[f"{metric}_mean"]
            reported = paper[metric][0]
            rows.append(
                {
                    "method": name,
                    "metric": metric,
                    "gap_pp": rep - reported,
                }
            )
    if not rows:
        print(f"No scores for gap plot: {category}")
        return

    gap_df = pd.DataFrame(rows)
    metrics_order = ["R@5", "NDCG@5", "R@10", "NDCG@10"]
    fig, ax = plt.subplots(figsize=(9, 4))
    bar_w = 0.35
    x = np.arange(len(metrics_order))
    for i, (name, color) in enumerate(
        [("SASRec++", "#4C72B0"), ("MARIUS (COSETTE)", "#DD8452")]
    ):
        part = gap_df[gap_df["method"] == name].set_index("metric").reindex(metrics_order)
        ax.bar(x + (i - 0.5) * bar_w, part["gap_pp"], width=bar_w, label=name, color=color)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xticks(x, metrics_order)
    ax.set_ylabel("Replication − paper (pp)")
    ax.set_title(f"{category}, test set")
    ax.legend()
    fig.tight_layout()
    plt.show()


def plot_seed_spread(category: str) -> None:
    slug = CATEGORIES[category]["slug"]
    configs = [
        ("SASRec++", f"sasrec_{slug}_5seed_full_scores.jsonl"),
        ("MARIUS (COSETTE)", f"marius_{slug}_5seed_full_scores.jsonl"),
    ]
    data, labels = [], []
    for name, fname in configs:
        records = load_scores_jsonl(RESULTS_DIR / fname)
        if not records:
            continue
        data.append([float(r["R@10"]) for r in records])
        labels.append(name)
    if not data:
        return

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.boxplot(data, tick_labels=labels)
    ax.set_ylabel("Test R@10 (%)")
    ax.set_title(f"{category}: seed spread (test R@10)")
    fig.tight_layout()
    plt.show()


def plot_sasrec_vs_marius_bar(category: str) -> None:
    slug = CATEGORIES[category]["slug"]
    configs = [
        ("SASRec++", f"sasrec_{slug}_5seed_full_scores.jsonl"),
        ("MARIUS (COSETTE)", f"marius_{slug}_5seed_full_scores.jsonl"),
    ]
    fig, ax = plt.subplots(figsize=(6, 4))
    labels, means, stds = [], [], []
    for name, fname in configs:
        summary = summarize_method(load_scores_jsonl(RESULTS_DIR / fname))
        if summary is None:
            continue
        labels.append(name)
        means.append(summary["R@10_mean"])
        stds.append(summary["R@10_std"])
    if not labels:
        return
    x = np.arange(len(labels))
    ax.bar(x, means, yerr=stds, capsize=4, color=["#4C72B0", "#DD8452"][: len(labels)])
    ax.set_xticks(x, labels)
    ax.set_ylabel("Test R@10 (%)")
    ax.set_title(f"{category}: SASRec++ vs MARIUS (test R@10)")
    fig.tight_layout()
    plt.show()


for _category in CATEGORIES:
    plot_averaged_sasrec_vs_marius(_category)
    plot_replication_gap(_category)
    plot_seed_spread(_category)
    plot_sasrec_vs_marius_bar(_category)

## 4. Learning curves (`reports/metrics/*.csv`)

Curves from `reports/metrics/*.csv`. Regenerate with:

`python scripts/export_metrics_for_report.py --source wandb`

In [ ]:
metrics_df = load_metrics_csvs()
if metrics_df.empty:
    print(
        f"No CSV files in {METRICS_DIR}.\n"
        "Export learning curves with:\n"
        "  python scripts/export_metrics_for_report.py --source wandb\n"
        "Then re-run this notebook."
    )
else:
    print(f"Loaded {len(metrics_df):,} metric rows from {metrics_df['source_file'].nunique()} files.")
    display(metrics_df.head())

In [ ]:
def plot_val_curves(category: str, method: str) -> None:
    if metrics_df.empty:
        return

    subset = metrics_df[
        (metrics_df["method"] == method)
        & (metrics_df["category"] == category)
        & (metrics_df["metric"].isin(["HR@10", "NDCG@10"]))
    ]
    if subset.empty:
        print(f"No validation curves for {method} / {category}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["HR@10", "NDCG@10"]):
        for seed, group in subset[subset["metric"] == metric].groupby("seed"):
            group = group.sort_values("step")
            ax.plot(group["step"], 100.0 * group["value"], label=f"seed {seed}", alpha=0.85)
        ax.set_xlabel("Step")
        ax.set_ylabel(f"valid {metric} (%)")
        ax.set_title(metric)
        ax.legend(fontsize=8)
    fig.suptitle(f"{method}, {category} (validation)")
    fig.tight_layout()
    plt.show()


if not metrics_df.empty:
    for category in CATEGORIES:
        plot_val_curves(category, "sasrec")
        plot_val_curves(category, "marius")

In [ ]:
def plot_cosette_curves() -> None:
    if metrics_df.empty:
        return
    subset = metrics_df[
        (metrics_df["method"] == "cosette")
        & (metrics_df["metric"].isin(["train_loss", "collision_rate"]))
    ]
    if subset.empty:
        print("No COSETTE curves exported yet.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, metric in zip(axes, ["train_loss", "collision_rate"]):
        for run_name, group in subset[subset["metric"] == metric].groupby("run_name"):
            group = group.sort_values("step")
            ax.plot(group["step"], group["value"], label=run_name, alpha=0.85)
        ax.set_xlabel("Epoch")
        ax.set_title(metric)
        ax.legend(fontsize=8)
    fig.suptitle("COSETTE training")
    fig.tight_layout()
    plt.show()


plot_cosette_curves()

## 5. Setup

- Test R@K and NDCG@K, mean ± std over seeds 42–46.
- Scores: `reports/results/*_scores.jsonl`. Curves: `reports/metrics/*.csv`.
- SASRec++ baseline; MARIUS with COSETTE (`experiment=marius_small`, 80k steps, batch 256, validation every 10k steps). Details in `REPRODUCIBILITY.md`.